# bridge vs. other sample rate converters — measured

The async engine's comparison notebook asks what clock recovery costs. This
one asks the synchronous question: at the one ratio pair `bridge` exists for,
**44.1 ↔ 48 kHz, both directions**, how do the general-purpose resampler
libraries measure against it — in quality, through the **same AES17-style
instrument applied identically to every subject**, and in the settings each
library needs to reach `bridge`'s two tiers. Every number here is measured
through the **shipping C++** (the C ABI via `tap_sr_bridge_py.py`) and the
real libraries (libsamplerate through its Python binding; soxr, r8brain and
SpeexDSP through the family's C shims in `tools/compare/shim`), never a
Python re-implementation.

The subjects, converting the same signals in the same direction:

| Subject | What it is | Settings measured |
|---|---|---|
| `bridge` | this engine, 147/160 polyphase, one clock | `economy` (70 dB, 18 kHz), `balanced`, `transparent` (120 dB, 20 kHz); `economy` and `transparent` in Q15 |
| libsamplerate | resampler library, five fixed converters | `SINC_FASTEST`, `SINC_MEDIUM`, `SINC_BEST` |
| soxr | resampler library, FFT-based, custom quality spec | 16-bit at an 18 kHz passband (custom), `HQ`, `VHQ` |
| r8brain-free-src | resampler library, FFT block convolution, continuous knobs | 70 dB at a 12 % band, 120 dB at a 6 % band, `CDSPResampler24` |
| SpeexDSP | resampler library, polyphase, the one with a fixed-point build | quality 3, 9, 10 (float); 3, 10 (fixed-point) |

**The matching rule** (stated once, applied by the sweep below): a
competitor's *matched* setting for a tier is its cheapest whose measured
stopband is at least the tier's (70 dB for `economy`, 120 dB for
`transparent`) and whose passband droops no more than 0.1 dB at the tier's
edge (18 kHz, 20 kHz), in **both** directions; its *frontier* is its best.
`economy` and `transparent` are what `bridge` ships; the competitors' knobs
are set from the measurement, not from their documentation.

In [1]:
import ctypes, pathlib, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
import figure_digest  # prints a data digest per figure (A/B gates)
figure_digest.install()
import samplerate
from tap_sr_bridge_py import RatioConverter, version

DOWN, UP = (48000.0, 44100.0), (44100.0, 48000.0)   # (input rate, output rate)
DIRECTIONS = {"48 → 44.1": DOWN, "44.1 → 48": UP}

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
TOOLS_DIR = REPO / "build"

def _find_dso(stem):
    for name in (f"lib{stem}.so", f"lib{stem}.dylib", f"{stem}.dll"):
        hits = sorted(TOOLS_DIR.rglob(name))
        if hits:
            return hits[0]
    return None

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout); print(r.stderr, file=sys.stderr)
        raise RuntimeError("command failed: " + " ".join(cmd))

# The family's C shims over the pinned r8brain-free-src and SpeexDSP sources
# and the system soxr (tools/compare/shim): r8brain and SpeexDSP have no
# maintained Python binding, and python-soxr exposes only the named recipes,
# not the custom quality spec the matched row needs. Always (re)build: an
# incremental build is quick, and a library left over from an older checkout
# would silently measure old code.
_run(["cmake", "-B", str(TOOLS_DIR), "-S", str(REPO), "-DCMAKE_BUILD_TYPE=Release",
      "-DTAP_SR_BUILD_COMPARE_SHIM=ON", "-DTAP_SR_BUILD_TESTS=OFF", "-DTAP_SR_BUILD_EXAMPLES=OFF"])
_run(["cmake", "--build", str(TOOLS_DIR), "--target", "tap_sr_cmp_r8b_shim", "tap_sr_cmp_speex_shim",
      "tap_sr_cmp_soxr_shim", "--config", "Release", "-j"])
R8B_DSO, SPEEX_DSO, SOXR_DSO = (_find_dso(s) for s in ("tap_sr_cmp_r8b_shim", "tap_sr_cmp_speex_shim", "tap_sr_cmp_soxr_shim"))
assert R8B_DSO and SPEEX_DSO and SOXR_DSO
_FP = ctypes.POINTER(ctypes.c_float)

_r8b = ctypes.CDLL(str(R8B_DSO))
_r8b.tap_sr_cmp_r8b_oneshot.restype = ctypes.c_int
_r8b.tap_sr_cmp_r8b_oneshot.argtypes = [_FP, ctypes.c_int, ctypes.c_double, ctypes.c_double, ctypes.c_double,
                                        ctypes.c_double, ctypes.c_int, _FP, ctypes.c_int]
_r8b.tap_sr_cmp_r8b_latency_frames.restype = ctypes.c_int
_r8b.tap_sr_cmp_r8b_latency_frames.argtypes = [ctypes.c_double] * 4 + [ctypes.c_int]
_speex = ctypes.CDLL(str(SPEEX_DSO))
_speex.tap_sr_cmp_speex_oneshot.restype = ctypes.c_int
_speex.tap_sr_cmp_speex_oneshot.argtypes = [_FP, ctypes.c_int, ctypes.c_double, ctypes.c_double, ctypes.c_int,
                                            ctypes.c_int, _FP, ctypes.c_int]
_speex.tap_sr_cmp_speex_latency_frames.restype = ctypes.c_int
_speex.tap_sr_cmp_speex_latency_frames.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.c_int, ctypes.c_int]
_soxr = ctypes.CDLL(str(SOXR_DSO))
_soxr.tap_sr_cmp_soxr_oneshot.restype = ctypes.c_int
_soxr.tap_sr_cmp_soxr_oneshot.argtypes = [_FP, ctypes.c_int, ctypes.c_double, ctypes.c_double, ctypes.c_int,
                                          ctypes.c_double, ctypes.c_double, _FP, ctypes.c_int]
_soxr.tap_sr_cmp_soxr_delay_frames.restype = ctypes.c_double
_soxr.tap_sr_cmp_soxr_delay_frames.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.c_int, ctypes.c_double, ctypes.c_double]
SOXR_HQ, SOXR_VHQ = 4, 6   # soxr.h: SOXR_20_BITQ, SOXR_28_BITQ

def _nout(n, src, dst):
    return int(round(n * dst / src))

# --- one whole-buffer conversion per subject: mono float32 in, float32 out ---
def via_bridge(x, src, dst, profile="economy", fmt="float"):
    conv = RatioConverter(direction="down" if src > dst else "up", profile=profile, fmt=fmt)
    if fmt == "q15":   # a Q15 caller's interface: round and saturate in, scale out
        q = np.clip(np.round(np.asarray(x, np.float64) * 32767.0), -32768, 32767).astype(np.int16)
        return np.concatenate([conv.process(q), conv.flush()]).astype(np.float32) / 32768.0
    return np.concatenate([conv.process(np.asarray(x, np.float32)), conv.flush()])

def via_lsr(x, src, dst, converter):
    return samplerate.resample(np.asarray(x, np.float32), dst / src, converter).astype(np.float32)

def via_soxr(x, src, dst, recipe=-1, precision=0.0, passband_end=0.0):
    x = np.ascontiguousarray(x, np.float32)
    y = np.zeros(_nout(len(x), src, dst) + 16, np.float32)
    n = _soxr.tap_sr_cmp_soxr_oneshot(x.ctypes.data_as(_FP), len(x), src, dst, recipe, precision, passband_end,
                                      y.ctypes.data_as(_FP), len(y))
    if n <= 0:
        raise ValueError(f"soxr rejected recipe={recipe} precision={precision} passband_end={passband_end}")
    return y[:n]

def via_r8brain(x, src, dst, trans_band_pct, atten_db):
    x = np.ascontiguousarray(x, np.float32)
    y = np.zeros(_nout(len(x), src, dst), np.float32)
    assert _r8b.tap_sr_cmp_r8b_oneshot(x.ctypes.data_as(_FP), len(x), src, dst, trans_band_pct, atten_db, 0,
                                       y.ctypes.data_as(_FP), len(y)) == 0
    return y

def via_speex(x, src, dst, quality, fixed_point=False):
    x = np.ascontiguousarray(x, np.float32)
    y = np.zeros(_nout(len(x), src, dst), np.float32)
    assert _speex.tap_sr_cmp_speex_oneshot(x.ctypes.data_as(_FP), len(x), src, dst, quality, int(fixed_point),
                                           y.ctypes.data_as(_FP), len(y)) == 0
    return y

# --- streaming latency, in input frames before the first output frame ---
def lsr_latency_frames(src, dst, converter):
    rs = samplerate.Resampler(converter, channels=1)
    fed = 0
    while fed < 100000:
        out = rs.process(np.zeros(1, np.float32), dst / src, end_of_input=False)
        fed += 1
        if len(out) > 0:
            return fed
    raise RuntimeError("no output")

def bridge_latency_frames(src, dst, profile="economy"):
    return RatioConverter(direction="down" if src > dst else "up", profile=profile).latency_input_frames

print(f"bridge C ABI version {version()}; shims: {R8B_DSO.name}, {SPEEX_DSO.name}, {SOXR_DSO.name}; "
      f"libsamplerate via samplerate {samplerate.__version__}")

bridge C ABI version (0, 6, 0); shims: libtap_sr_cmp_r8b_shim.so, libtap_sr_cmp_speex_shim.so, libtap_sr_cmp_soxr_shim.so; libsamplerate via samplerate 0.2.4


## The instrument, and its calibration

The same AES17-style measurement as the async comparison: THD+N is a
997 Hz tone at −1 dBFS with the fundamental removed by an exact
least-squares fit at the phase-slope-refined frequency plus a ±20 Hz notch,
the residual integrated 20 Hz–20 kHz; dynamic range is the same at −60 dBFS,
A-weighted. The **24-bit interface** columns quantize every subject's output
to 24 bits, the condition under which software numbers are comparable to
silicon datasheets. The instrument is validated against synthetic signals
with known answers at both output rates before it measures anything.

In [2]:
def a_weight(f):
    f2 = np.square(f)
    num = (12194.0**2) * f2 * f2
    den = ((f2 + 20.6**2) * np.sqrt((f2 + 107.7**2) * (f2 + 737.9**2)) * (f2 + 12194.0**2))
    ra = np.divide(num, den, out=np.zeros_like(f2), where=den > 0)
    ref = (12194.0**2 * 1000.0**4) / ((1000.0**2 + 20.6**2)
          * np.sqrt((1000.0**2 + 107.7**2) * (1000.0**2 + 737.9**2)) * (1000.0**2 + 12194.0**2))
    return ra / ref

def refine_freq(x, f0, fs, win=8192, iters=3):
    n = (len(x) // win) * win
    tt = np.arange(n)
    for _ in range(iters):
        ph = []
        for k in range(n // win):
            sl = slice(k * win, (k + 1) * win)
            A = np.column_stack([np.sin(2 * np.pi * f0 / fs * tt[sl]), np.cos(2 * np.pi * f0 / fs * tt[sl])])
            c, *_ = np.linalg.lstsq(A, x[sl].astype(np.float64), rcond=None)
            ph.append(np.arctan2(c[1], c[0]))
        ph = np.unwrap(ph)
        tk = (np.arange(len(ph)) + 0.5) * win / fs
        f0 += np.polyfit(tk, ph, 1)[0] / (2 * np.pi)
    return f0

def thdn_db(x, f0_nominal, fs, weighted=False, notch_hz=20.0, band=(20.0, 20000.0)):
    # AES17-style THD+N in dB relative to the fundamental (negative).
    x = x.astype(np.float64)
    f0 = refine_freq(x, f0_nominal, fs)
    tt = np.arange(len(x))
    A = np.column_stack([np.sin(2 * np.pi * f0 / fs * tt), np.cos(2 * np.pi * f0 / fs * tt)])
    c, *_ = np.linalg.lstsq(A, x, rcond=None)
    fund_rms = np.hypot(c[0], c[1]) / np.sqrt(2)
    resid = x - A @ c
    R = np.fft.rfft(resid)
    f = np.fft.rfftfreq(len(resid), 1 / fs)
    mask = (f >= band[0]) & (f <= band[1]) & (np.abs(f - f0) > notch_hz)
    w = a_weight(f) if weighted else np.ones_like(f)
    resid_rms = np.sqrt(np.sum(np.abs(R[mask] * w[mask])**2) * 2 / len(resid)**2)
    return 20 * np.log10(resid_rms / fund_rms)

def fit_amp(y, f, fs):
    # Amplitude of the component at a known frequency (least squares).
    y = np.asarray(y, np.float64)
    t = np.arange(len(y))
    A = np.column_stack([np.sin(2 * np.pi * f / fs * t), np.cos(2 * np.pi * f / fs * t)])
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    return np.hypot(c[0], c[1])

def mid(y, fs, seconds):
    # The middle of the stream: one-shot converters flush a filter tail at
    # the end, and the start carries the delay they remove.
    n = int(seconds * fs)
    s = (len(y) - n) // 2
    return np.asarray(y[s:s + n], np.float32)

def q24(y):
    # Round to a 24-bit interface, undithered -- what silicon presents at its pins.
    return np.round(np.asarray(y, np.float64) * 8388608.0) / 8388608.0

def tone(f, seconds, fs, amp_dbfs):
    return (10 ** (amp_dbfs / 20) * np.sin(2 * np.pi * f / fs * np.arange(int(seconds * fs)))).astype(np.float32)

rng = np.random.default_rng(11)
for fs in (44100.0, 48000.0):
    n = int(8 * fs)
    sine = tone(997.003, 8.0, fs, -1.0)
    for noise_db in (-100.0, -130.0):
        sig = (sine + 10 ** (noise_db / 20) * rng.standard_normal(n)).astype(np.float32)
        got = thdn_db(sig, 997.0, fs)
        inband = (20000.0 - 20.0) / (fs / 2)
        expect = noise_db - 20 * np.log10(10 ** (-1 / 20) / np.sqrt(2)) + 10 * np.log10(inband)
        print(f"fs {fs:.0f}: injected {noise_db:6.1f} dBFS noise -> measured THD+N {got:7.2f} dB (expect {expect:7.2f})")
        assert abs(got - expect) < 0.5
print("instrument calibrated at both output rates")

fs 44100: injected -100.0 dBFS noise -> measured THD+N  -96.44 dB (expect  -96.42)


fs 44100: injected -130.0 dBFS noise -> measured THD+N -126.42 dB (expect -126.42)


fs 48000: injected -100.0 dBFS noise -> measured THD+N  -96.78 dB (expect  -96.79)


fs 48000: injected -130.0 dBFS noise -> measured THD+N -126.76 dB (expect -126.79)
instrument calibrated at both output rates


## The matched settings, measured

Each library's quality knob is swept in both directions. Two numbers per
setting decide the match: the **passband droop** at the tier's edge (the
fitted gain of an 18 kHz and a 20 kHz tone, relative to 997 Hz) and the
**worst spurious product**, measured at the exact frequencies the arithmetic
puts them: going down (48 → 44.1), aliases of ultrasonic input tones
(22.3, 23, 23.9 kHz → 44.1 − f) and images of in-band tones
(16–18 kHz → f − 3.9 kHz, the 48 − f image folded about 44.1); going up
(44.1 → 48), images of in-band tones (16–18 kHz → f + 3.9 kHz). The rule
then picks, per tier, the cheapest setting that holds both in both
directions. The picks are asserted against the constants the host and
embedded benches compile in, so the benches cannot drift from this
measurement.

In [3]:
def droop_db(conv, src, dst, f):
    y = mid(conv(tone(f, 1.5, src, -6.0), src, dst), dst, 1.0)
    ref = mid(conv(tone(997.0, 1.5, src, -6.0), src, dst), dst, 1.0)
    return 20 * np.log10(fit_amp(y, f, dst) / fit_amp(ref, 997.0, dst))

def worst_product_db(conv, src, dst):
    if src > dst:
        probes = [(22300.0, 44100.0 - 22300.0), (23000.0, 44100.0 - 23000.0), (23900.0, 44100.0 - 23900.0),
                  (16000.0, 12100.0), (17000.0, 13100.0), (18000.0, 14100.0)]
    else:
        probes = [(16000.0, 19900.0), (17000.0, 20900.0), (18000.0, 21900.0)]
    worst = -400.0
    for f, fp in probes:
        y = mid(conv(tone(f, 1.5, src, -6.0), src, dst), dst, 1.0)
        worst = max(worst, 20 * np.log10(fit_amp(y, fp, dst) / 10 ** (-6 / 20)))
    return worst

# Candidate settings per library, cheapest first (the order the rule picks in).
CANDIDATES = {
    "libsamplerate": [(c, lambda x, s, d, c=c: via_lsr(x, s, d, c)) for c in ("sinc_fastest", "sinc_medium", "sinc_best")],
    "soxr": [(f"{p:.0f}-bit @ {pe:.3f}", lambda x, s, d, p=p, pe=pe: via_soxr(x, s, d, -1, p, pe))
             for p in (16.0, 20.0) for pe in (0.816, 0.840, 0.862, 0.907)]
            + [("HQ", lambda x, s, d: via_soxr(x, s, d, SOXR_HQ)), ("VHQ", lambda x, s, d: via_soxr(x, s, d, SOXR_VHQ))],
    "r8brain": [(f"{a:.0f} dB @ {tb:g} %", lambda x, s, d, a=a, tb=tb: via_r8brain(x, s, d, tb, a))
                for a, tbs in ((70.0, (18.4, 13.8, 12.0, 11.0, 10.0, 9.3)), (120.0, (9.3, 8.0, 6.0, 5.0, 4.0, 3.0, 2.0)))
                for tb in tbs],
    "SpeexDSP float": [(f"q{q}", lambda x, s, d, q=q: via_speex(x, s, d, q)) for q in range(2, 11)],
    "SpeexDSP fixed": [(f"q{q}", lambda x, s, d, q=q: via_speex(x, s, d, q, True)) for q in (2, 3, 4, 9, 10)],
}
TIERS = {"economy": (70.0, 18000.0), "transparent": (120.0, 20000.0)}   # (stopband dB, passband edge Hz)
DROOP_MAX_DB = 0.1

sweep = {}   # (library, setting) -> per direction: (droop18, droop20, worst)
for lib, cands in CANDIDATES.items():
    print(f"--- {lib}")
    print(f"{'setting':18s} " + "   ".join(f"{d:>9s}: 18k   20k   worst" for d in DIRECTIONS))
    for name, conv in cands:
        row = {}
        for dname, (src, dst) in DIRECTIONS.items():
            try:
                row[dname] = (droop_db(conv, src, dst, 18000.0), droop_db(conv, src, dst, 20000.0),
                              worst_product_db(conv, src, dst))
            except ValueError:
                row[dname] = None
        sweep[(lib, name)] = row
        print(f"{name:18s} " + "   ".join(("   rejected by soxr    " if r is None else
                                           f"{r[0]:7.2f} {r[1]:6.2f} {r[2]:7.1f}") for r in row.values()))

def pick(lib, tier):
    atten, edge = TIERS[tier]
    for name, _ in CANDIDATES[lib]:
        rows = sweep[(lib, name)].values()
        if all(r is not None and -r[2] >= atten and abs(r[0] if edge == 18000.0 else r[1]) <= DROOP_MAX_DB for r in rows):
            return name
    return None

MATCHED = {tier: {lib: pick(lib, tier) for lib in CANDIDATES} for tier in TIERS}
for tier, picks in MATCHED.items():
    print(f"{tier}-matched:", ", ".join(f"{lib} = {p}" for lib, p in picks.items()))
# The benches' constants (bench/compare/bench_compare.cpp, bench/icount/cmp_main.cpp).
assert MATCHED["economy"] == {"libsamplerate": "sinc_medium", "soxr": "16-bit @ 0.816", "r8brain": "70 dB @ 12 %",
                              "SpeexDSP float": "q3", "SpeexDSP fixed": "q3"}, MATCHED["economy"]
assert MATCHED["transparent"]["libsamplerate"] == "sinc_best"
assert MATCHED["transparent"]["soxr"] in ("20-bit @ 0.907", "HQ")
assert MATCHED["transparent"]["r8brain"] == "120 dB @ 6 %"
assert MATCHED["transparent"]["SpeexDSP float"] == "q9"
assert MATCHED["transparent"]["SpeexDSP fixed"] is None   # 16-bit arithmetic cannot reach 120 dB

--- libsamplerate
setting            48 → 44.1: 18k   20k   worst   44.1 → 48: 18k   20k   worst


sinc_fastest         -4.26 -22.82  -105.1     -4.26 -22.82  -105.7


sinc_medium          -0.00  -3.25  -123.2     -0.00  -3.25  -136.5


sinc_best            -0.00  -0.00  -156.5     -0.00  -0.00  -174.3
--- soxr
setting            48 → 44.1: 18k   20k   worst   44.1 → 48: 18k   20k   worst
16-bit @ 0.816       -0.00  -5.79  -110.0     -0.01  -5.80  -122.7
16-bit @ 0.840       -0.00  -3.04  -109.7     -0.01  -3.06  -122.0


16-bit @ 0.862       -0.00  -1.12  -111.9     -0.01  -1.15  -123.7
16-bit @ 0.907       -0.00  -0.01  -114.3     -0.00  -0.00  -125.6
20-bit @ 0.816       -0.00  -5.77  -137.6     -0.00  -5.77  -140.0


...

q10                  -0.00  -0.00  -123.4     -0.00  -0.00  -131.9
--- SpeexDSP fixed
setting            48 → 44.1: 18k   20k   worst   44.1 → 48: 18k   20k   worst
q2                   -0.51 -11.05   -66.2     -0.19  -5.65   -69.8


q3                   -0.06  -8.69   -78.0     -0.01  -4.40   -84.4


q4                   -0.00  -3.18   -75.7     -0.00  -1.11   -90.0


q9                   -0.00  -0.00   -81.6     -0.00  -0.00   -84.1


q10                  -0.00  -0.00   -84.9     -0.00  -0.00   -90.9
economy-matched: libsamplerate = sinc_medium, soxr = 16-bit @ 0.816, r8brain = 70 dB @ 12 %, SpeexDSP float = q3, SpeexDSP fixed = q3
transparent-matched: libsamplerate = sinc_best, soxr = 20-bit @ 0.907, r8brain = 120 dB @ 6 %, SpeexDSP float = q9, SpeexDSP fixed = None


## THD+N and dynamic range at the 24-bit interface, both directions

997 Hz at −1 dBFS; DR at −60 dBFS, A-weighted. Every subject converts the
same stream in the same direction; the libraries are handed the exact ratio
(which, unlike the async case, is no favour here: the ratio *is* exact for
everyone). The two fixed-point rows are the 16-bit interface's floor, as the
async comparison found: a Q15 caller gets the same quality from `bridge`'s
Q15 converters and from SpeexDSP's fixed-point build.

In [4]:
SUBJECTS = [
    ("bridge economy",               lambda x, s, d: via_bridge(x, s, d, "economy")),
    ("bridge balanced",              lambda x, s, d: via_bridge(x, s, d, "balanced")),
    ("bridge transparent",           lambda x, s, d: via_bridge(x, s, d, "transparent")),
    ("libsamplerate FASTEST",        lambda x, s, d: via_lsr(x, s, d, "sinc_fastest")),
    ("libsamplerate MEDIUM (eco)",   lambda x, s, d: via_lsr(x, s, d, "sinc_medium")),
    ("libsamplerate BEST (tr)",      lambda x, s, d: via_lsr(x, s, d, "sinc_best")),
    ("soxr 16-bit @ 0.816 (eco)",    lambda x, s, d: via_soxr(x, s, d, -1, 16.0, 0.816)),
    ("soxr HQ (tr)",                 lambda x, s, d: via_soxr(x, s, d, SOXR_HQ)),
    ("soxr VHQ",                     lambda x, s, d: via_soxr(x, s, d, SOXR_VHQ)),
    ("r8brain 70 dB @ 12 % (eco)",   lambda x, s, d: via_r8brain(x, s, d, 12.0, 70.0)),
    ("r8brain 120 dB @ 6 % (tr)",    lambda x, s, d: via_r8brain(x, s, d, 6.0, 120.0)),
    ("r8brain CDSPResampler24",      lambda x, s, d: via_r8brain(x, s, d, 2.0, 180.15)),
    ("SpeexDSP q3 float (eco)",      lambda x, s, d: via_speex(x, s, d, 3)),
    ("SpeexDSP q9 float (tr)",       lambda x, s, d: via_speex(x, s, d, 9)),
    ("SpeexDSP q10 float",           lambda x, s, d: via_speex(x, s, d, 10)),
    ("bridge economy Q15",           lambda x, s, d: via_bridge(x, s, d, "economy", "q15")),
    ("bridge transparent Q15",       lambda x, s, d: via_bridge(x, s, d, "transparent", "q15")),
    ("SpeexDSP q3 fixed-point",      lambda x, s, d: via_speex(x, s, d, 3, True)),
    ("SpeexDSP q10 fixed-point",     lambda x, s, d: via_speex(x, s, d, 10, True)),
]
ANALYZE = 4.0
results = {}   # direction -> name -> (thdn float, thdn 24-bit, dr float, dr 24-bit)
for dname, (src, dst) in DIRECTIONS.items():
    x_full = tone(997.0, ANALYZE + 2.0, src, -1.0)
    x_dr = tone(997.0, ANALYZE + 2.0, src, -60.0)
    results[dname] = {}
    print(f"===== {dname} kHz")
    print(f"{'subject':30s} {'THD+N fl':>9s} {'THD+N 24b':>10s} {'DR fl':>7s} {'DR 24b':>7s}")
    for name, conv in SUBJECTS:
        y = mid(conv(x_full, src, dst), dst, ANALYZE)
        yd = mid(conv(x_dr, src, dst), dst, ANALYZE)
        r = (thdn_db(y, 997.0, dst), thdn_db(q24(y), 997.0, dst),
             -thdn_db(yd, 997.0, dst, weighted=True) + 60.0, -thdn_db(q24(yd), 997.0, dst, weighted=True) + 60.0)
        results[dname][name] = r
        print(f"{name:30s} {r[0]:9.1f} {r[1]:10.1f} {r[2]:7.1f} {r[3]:7.1f}")

# bridge's own rows, pinned (24-bit THD+N; the acceptance numbers in PLAN.md section 8 are the design's,
# these are the shipping engine's through the C ABI).
for dname in DIRECTIONS:
    r = results[dname]
    assert r["bridge economy"][1] < -85.0, r["bridge economy"]
    assert r["bridge transparent"][1] < -138.0, r["bridge transparent"]
    assert r["bridge economy Q15"][1] < -74.0, r["bridge economy Q15"]
    assert r["SpeexDSP q3 fixed-point"][1] < -80.0

===== 48 → 44.1 kHz
subject                         THD+N fl  THD+N 24b   DR fl  DR 24b


bridge economy                     -91.4      -91.4   151.2   146.7


bridge balanced                    -89.3      -89.3   149.1   145.9


bridge transparent                -145.0     -141.7   205.2   148.8


libsamplerate FASTEST             -103.4     -103.4   170.6   148.7


...

SpeexDSP q10 float                -137.5     -136.7   197.6   149.1


bridge economy Q15                 -78.9      -78.9    97.9    97.9


bridge transparent Q15             -77.5      -77.5    97.9    97.9


SpeexDSP q3 fixed-point            -89.9      -89.9    97.8    97.8


SpeexDSP q10 fixed-point           -79.3      -79.3    97.8    97.8


## Latency

Input frames each engine consumes before its first output frame: `bridge`'s
is its filter group delay (`latency_input_frames()`); libsamplerate's is
counted through its streaming API; r8brain's is `getInLenBeforeOutPos(0)`;
SpeexDSP's is `get_input_latency()`; soxr's is `soxr_delay()` after priming,
which it reports in *output* frames (labelled). The milliseconds are at the
input rate.

In [5]:
LATENCY = [
    ("bridge economy",            lambda s, d: bridge_latency_frames(s, d, "economy")),
    ("bridge transparent",        lambda s, d: bridge_latency_frames(s, d, "transparent")),
    ("libsamplerate FASTEST",     lambda s, d: lsr_latency_frames(s, d, "sinc_fastest")),
    ("libsamplerate MEDIUM (eco)", lambda s, d: lsr_latency_frames(s, d, "sinc_medium")),
    ("libsamplerate BEST (tr)",   lambda s, d: lsr_latency_frames(s, d, "sinc_best")),
    ("soxr 16-bit @ 0.816 (eco)*", lambda s, d: _soxr.tap_sr_cmp_soxr_delay_frames(s, d, -1, 16.0, 0.816)),
    ("soxr HQ (tr)*",             lambda s, d: _soxr.tap_sr_cmp_soxr_delay_frames(s, d, SOXR_HQ, 0.0, 0.0)),
    ("soxr VHQ*",                 lambda s, d: _soxr.tap_sr_cmp_soxr_delay_frames(s, d, SOXR_VHQ, 0.0, 0.0)),
    ("r8brain 70 dB @ 12 % (eco)", lambda s, d: _r8b.tap_sr_cmp_r8b_latency_frames(s, d, 12.0, 70.0, 0)),
    ("r8brain 120 dB @ 6 % (tr)", lambda s, d: _r8b.tap_sr_cmp_r8b_latency_frames(s, d, 6.0, 120.0, 0)),
    ("r8brain CDSPResampler24",   lambda s, d: _r8b.tap_sr_cmp_r8b_latency_frames(s, d, 2.0, 180.15, 0)),
    ("SpeexDSP q3 (eco)",         lambda s, d: _speex.tap_sr_cmp_speex_latency_frames(s, d, 3, 0)),
    ("SpeexDSP q9 (tr)",          lambda s, d: _speex.tap_sr_cmp_speex_latency_frames(s, d, 9, 0)),
    ("SpeexDSP q10",              lambda s, d: _speex.tap_sr_cmp_speex_latency_frames(s, d, 10, 0)),
]
latency = {}
print(f"{'subject (* = output frames)':30s} " + "   ".join(f"{d:>18s}" for d in DIRECTIONS))
for name, fn in LATENCY:
    latency[name] = {}
    cells = []
    for dname, (src, dst) in DIRECTIONS.items():
        fr = float(fn(src, dst))
        rate = dst if name.endswith("*") else src
        latency[name][dname] = (fr, 1000.0 * fr / rate)
        cells.append(f"{fr:7.1f} ({1000.0 * fr / rate:5.2f} ms)")
    print(f"{name:30s} " + "   ".join(cells))
assert latency["bridge economy"]["48 → 44.1"][0] < 30 and latency["bridge economy"]["44.1 → 48"][0] < 20

subject (* = output frames)             48 → 44.1            44.1 → 48
bridge economy                    29.0 ( 0.60 ms)      19.0 ( 0.43 ms)
bridge transparent                92.0 ( 1.92 ms)      48.0 ( 1.09 ms)
libsamplerate FASTEST             23.0 ( 0.48 ms)      21.0 ( 0.48 ms)
libsamplerate MEDIUM (eco)        52.0 ( 1.08 ms)      48.0 ( 1.09 ms)
libsamplerate BEST (tr)          158.0 ( 3.29 ms)     145.0 ( 3.29 ms)
soxr 16-bit @ 0.816 (eco)*       226.2 ( 5.13 ms)     209.2 ( 4.36 ms)
soxr HQ (tr)*                    818.2 (18.55 ms)     892.2 (18.59 ms)
soxr VHQ*                        555.2 (12.59 ms)     392.2 ( 8.17 ms)
r8brain 70 dB @ 12 % (eco)       105.0 ( 2.19 ms)     107.0 ( 2.43 ms)
r8brain 120 dB @ 6 % (tr)        429.0 ( 8.94 ms)     436.0 ( 9.89 ms)
r8brain CDSPResampler24         1668.0 (34.75 ms)    1700.0 (38.55 ms)
SpeexDSP q3 (eco)                 28.0 ( 0.58 ms)      24.0 ( 0.54 ms)
SpeexDSP q9 (tr)                 104.0 ( 2.17 ms)      96.0 ( 2.18 ms)
SpeexD

## Reading the results

- **One competitor offers `economy`'s design point; the rest overshoot it.**
  The rule asks each library for the cheapest setting that keeps an 18 kHz
  passband at 70 dB. r8brain at 70 dB and a 12 % band is a true match: its
  worst product measures 71–72 dB and its THD+N −88 dB, beside `economy`'s
  72–74 dB and −91 / −86 dB. The others cannot turn the extra attenuation
  off: libsamplerate's `FASTEST` (97 dB) droops 4 dB at 18 kHz, so `MEDIUM`
  (123 dB) is its economy row; soxr's lowest precision is 16-bit (~110 dB);
  SpeexDSP's quality 3 is 85 dB. `economy`'s 70 dB is the speed-first trade
  `bridge`'s charter makes (every product confined above 20 kHz by
  arithmetic); the cost tables in `docs/COMPARISON.md` say what each
  competitor pays for the attenuation it cannot decline.
- **At the `transparent` tier the matched rows are real matches.**
  libsamplerate `BEST`, soxr `HQ`, r8brain at 120 dB / 6 % and SpeexDSP at
  quality 9 all hold 20 kHz flat at ≥ 120 dB; `transparent` and the frontier
  settings (`BEST`, `VHQ`, `CDSPResampler24`) all measure at the 24-bit
  ceiling (−142 to −144 dB), and the 120 dB-class rows 8–10 dB above it.
  `transparent`'s −121 dB design figure bounds alias and image products,
  which a 997 Hz probe does not excite inside 20 Hz–20 kHz.
- **The 16-bit rows are format floors, as in the async comparison.** `bridge`
  economy Q15 and SpeexDSP's fixed-point build at quality 3 both sit at the
  16-bit interface's ~98 dB A-weighted DR and within 10 dB of each other in
  THD+N; `transparent` in Q15 is no better than `economy` in Q15, which is
  why the README pairs Q15 with the economy-class tiers.
- **`economy`'s A-weighted DR sits 2–5 dB under the 24-bit ceiling** (146.7
  and 144.0 dB against 148.7): its 70 dB imaging floor is what a −60 dBFS
  probe exposes. The 120 dB rows and every frontier row are at the ceiling.
- **Latency.** `bridge` economy consumes 29 input frames (0.6 ms) going down
  and 19 (0.4 ms) going up before its first output. Among the
  economy-matched competitors SpeexDSP's quality 3 is at the same latency,
  libsamplerate `MEDIUM` ~2×, r8brain ~4×, soxr ~8× (in output frames); the
  frontier settings are 19× (soxr `VHQ`) to 57× (`CDSPResampler24`).

The computational side — host throughput, and executed instructions per
output frame on Cortex-M55, Cortex-M33 and Hexagon with each engine's
construction split out — is in `docs/COMPARISON.md`, measured with the same
matched settings.